# Playlist Detective 2: Artist Bundles
### MATH 3280 · Frequent Itemsets: Association Rules and A-Priori

**Tunebasket**, a small music-streaming startup, has 1,000 user playlists. This time we treat each playlist as a **basket** of artists and ask: which artists tend to appear together? Good answers become recommendations: "people who add *X* and *Y* also add *Z*."

You will first measure association rules between pairs of artists by brute force, then build the A-Priori algorithm to find larger frequent itemsets efficiently and finish with rules like $\{X, Y\} \to Z$.

**Rules**

* Implement the rule measures and A-Priori **yourself**.
* Use the definitions from our notes. Support is a **count** of baskets, and an itemset is **frequent** if $Support(I) > s$ (strictly greater). Throughout, $s = 50$.

* Use only `numpy`, `pandas`, `matplotlib`, `itertools`, `collections`, and `time`. You may use a library such as `mlxtend` or `datasketch` *only* to double-check your answers, and you must say so if you do.
* Every cell marked `# TODO` needs code. Every "**Your answer**" cell needs a few sentences.
* Run the whole notebook top to bottom (*Kernel → Restart & Run All*) before you submit.

All artist names are fictional.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from collections import Counter, defaultdict
import time

SEED = 3280
rng = np.random.default_rng(SEED)

## Part 0 · Load and explore the data (10 pts)

`playlists.csv` has one row per (playlist, artist). Build:

* `baskets`: a dictionary `playlist_id -> {set of artists}`
* `playlist_ids`: a sorted list of playlist ids, and `artists`: a sorted list of all artists
* `basket_list`: a list of the baskets (sets) in `playlist_ids` order

In [ ]:
df = pd.read_csv("playlists.csv")

baskets = ...        # TODO: dict playlist_id -> {set of artists}
playlist_ids = ...   # TODO: sorted list of playlist ids
artists = ...        # TODO: sorted list of artists
basket_list = ...    # TODO: list of baskets in playlist_ids order
n_baskets = len(basket_list)
print("baskets:", n_baskets, " artists:", len(artists))

In [ ]:
# Quick look: playlist sizes and the most common artists
sizes = pd.Series({p: len(s) for p, s in baskets.items()})
print(sizes.describe())
print(df["artist"].value_counts().head(10))

**Q0.** How many different pairs of artists are possible? How many pairs does an average playlist (about 17 artists) contain? Why does that gap matter for how we store pair counts?

**Your answer:** *write here*

## Part 1 · Warm-up: the measures (15 pts)

Write the four helper functions below, then check them against the **8-basket example from the notes** (support threshold $s = 4$):

{A,B,C}, {B,C,F}, {B,C,D}, {C,D,E}, {B,C,E}, {A,C,D}, {B,E,F}, {B,C,E}

The `assert` cells must pass before you move on.

In [ ]:
def support(itemset, basket_list):
    # TODO: number of baskets that contain every item in itemset
    raise NotImplementedError

def confidence(I, j, basket_list):
    # TODO: Support(I ∪ {j}) / Support(I)
    raise NotImplementedError

def interest(I, j, basket_list):
    # TODO: Confidence(I -> j) - Support({j}) / (# of baskets)
    raise NotImplementedError

def lift(I, j, basket_list):
    # TODO: Confidence(I -> j) / (Support({j}) / # of baskets)
    raise NotImplementedError

In [ ]:
notes = [set(x) for x in ["ABC", "BCF", "BCD", "CDE", "BCE", "ACD", "BEF", "BCE"]]

assert support({"B"}, notes) == 6 and support({"C"}, notes) == 7
assert support({"B", "C"}, notes) == 5 and support({"B", "E"}, notes) == 3
assert abs(confidence({"B", "C"}, "E", notes) - 2/5) < 1e-9
assert abs(interest({"B", "C"}, "E", notes) - (-1/10)) < 1e-9
assert abs(lift({"B", "C"}, "E", notes) - 0.80) < 1e-9
print("Part 1 checks passed.")

## Part 2 · Association rules between pairs of artists (30 pts)

Before building A-Priori, find pair rules by brute force.

**2a.** Count every artist (`item_counts`) and every pair of artists that appears together in some playlist (`pair_counts`, a `Counter` keyed by sorted tuples). Time the pair count and report how many distinct pairs you stored.

**2b.** For every pair $\{x, y\}$ with $Support(\{x, y\}) > 50$, form **both** rules $x \to y$ and $y \to x$. Build a DataFrame with columns `I`, `j`, `support`, `confidence`, `interest`, `lift`. Use your counts, not the Part 1 functions (those rescan every basket and would be slow).

**2c.** Show the 10 rules with the highest lift among rules with confidence $\ge 0.5$.

**2d.** Show the 5 rules with the highest confidence whose consequent is `"Nova Lux"`, with their lift.

In [ ]:
S_THRESH = 50

item_counts = ...   # TODO: Counter of single artists
pair_counts = ...   # TODO: Counter of sorted artist pairs (time this step)

In [ ]:
pair_rules = []
# TODO 2b: one row per rule x -> y (and y -> x) for every pair with support > S_THRESH
pair_rules = pd.DataFrame(pair_rules)
# TODO 2c: top 10 by lift among rules with confidence >= 0.5

In [ ]:
# TODO 2d: rules whose consequent is "Nova Lux"

**Q2.** (a) Pick two of your top-lift rules and explain them to the Tunebasket product team in plain language. (b) Several rules into *Nova Lux* have confidence above 0.5. Should Tunebasket recommend Nova Lux based on them? Use **interest** and **lift** in your answer. (c) Why is confidence alone a poor way to rank rules?

**Your answer:** *write here*

## Part 3 · The A-Priori algorithm (40 pts)

Counting every pair worked here, but counting every **triple** the same way would mean storing far more counts, and real data has millions of items. A-Priori uses **monotonicity** (every subset of a frequent itemset is frequent) to count only what could possibly be frequent.

**3a.** Pass 1: count every artist and build $L_1$.

**3b.** Pass 2: for each basket, keep only frequent artists and count the pairs. Build $L_2$. Compare the number of pairs you counted with the number stored in Part 2a.

**3c.** Keep going. Build $C_{k+1}$ from $L_k$: a $(k{+}1)$-set is a candidate only if **all** of its $k$-subsets are in $L_k$. Count, filter, and stop when $L_k$ is empty. Return a dictionary `frequent` mapping `frozenset -> support`, the levels $L_k$, and the number of pairs counted in pass 2.

In [ ]:
def apriori(basket_list, s):
    # TODO: pass 1 -> L1, pass 2 -> L2, then C_{k+1} from L_k until L_k is empty
    # return: frequent (dict frozenset -> support), levels (dict k -> L_k), pairs_counted
    raise NotImplementedError

frequent, levels, pairs_counted = apriori(basket_list, S_THRESH)
for k, Lk in levels.items():
    print(f"|L{k}| = {len(Lk)}")

In [ ]:
# Checks: every frequent set really has support > s, monotonicity holds,
# and A-Priori agrees with your brute-force pair counts from Part 2
for fs, c in list(frequent.items())[:200]:
    assert c == support(fs, basket_list) and c > S_THRESH
for fs in frequent:
    for sub in combinations(fs, len(fs) - 1):
        if sub:
            assert frozenset(sub) in frequent
assert {frozenset(p) for p, c in pair_counts.items() if c > S_THRESH} == set(levels[2])
print("Part 3 checks passed.")

**3d. Rules from every frequent itemset.** For every frequent itemset $F$ with $|F| \ge 2$ and every $j \in F$, form $I \to j$ with $I = F \setminus \{j\}$. Use the supports stored in `frequent`; don't rescan the baskets. Show the 10 rules with the highest lift among rules with confidence $\ge 0.5$. How many of them have two artists on the left?

In [ ]:
rules = []
# TODO: one row per rule I -> j from every frequent itemset of size >= 2
rules = pd.DataFrame(rules)
# TODO: top 10 by lift among rules with confidence >= 0.5

**Q3.** Give the largest frequent itemsets you found. Explain, using monotonicity, why A-Priori never needed to count most of the $\binom{481}{3}$ triples. What do the $\{X, Y\} \to Z$ rules in 3d tell Tunebasket that the pair rules from Part 2 could not?

**Your answer:** *write here*

## Part 4 · Connect the two projects (5 pts + optional bonus)

In Playlist Detective 1 you found about 20 **copycat** pairs: playlists that are near-copies of each other.

**Q4.** How could those copies bias the supports you computed here? Describe how you would check.

**Bonus (+5).** Do the check. Find the pairs of playlists with Jaccard similarity $\ge 0.6$ (reuse your Project 1 results, or a brute-force search), drop one playlist from each pair, re-run A-Priori with $s = 50$, and report what changes in $L_1$, $L_2$, $L_3$ and your top-lift rules.

**Your answer:** *write here*

In [ ]:
# Bonus (optional)

## Submission checklist

* [ ] Notebook runs top to bottom with no errors
* [ ] All `assert` checks pass
* [ ] Every **Your answer** cell is filled in
* [ ] Any library used to double-check answers is named and discussed